In [ ]:
import torch
print(torch.cuda.is_available())

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install sentence-transformers openpyxl scikit-learn

In [4]:
import pandas as pd
df = pd.read_excel('/content/drive/MyDrive/LLM_odev2_veriler.xlsx')

for i in range(len(df)):
  check_valid = len(str(df.loc[i, 'CosmosLLM düşünme süreci']))
  if check_valid <= 20:
    df.drop(i, inplace=True)

In [5]:
import random as r

l = [1,2,3]
for i in range(3):
  r.seed("AIHW2")
  r.choice(l)

def split_Data():
  testing_place = r.sample(range(len(df)), 1000)
  testing_rows = df.iloc[testing_place]


split_Data()

In [6]:
score_change = {
    'çok iyi': 5,
    'iyi': 4,
    'orta': 3,
    'kötü': 2,
    'çok kötü': 1
}

def map_score(value):
    if isinstance(value, str):
        return score_change.get(value, value)
    return value

df['Değerlendirme Puanınız'] = df['Değerlendirme Puanınız'].apply(map_score)

print(df["Değerlendirme Puanınız"])

0        4
1        3
2        5
3        4
4        2
        ..
12129    3
12130    5
12131    3
12132    3
12133    2
Name: Değerlendirme Puanınız, Length: 10990, dtype: int64


In [7]:
#S, D, C, score = df['Sorunuz'], df['CosmosLLM düşünme süreci'], df['CosmosLLM cevabı'], df['Değerlendirme Puanınız']
#print(df.columns)
df.columns = ['S', 'D', 'C', 'score']
print(df.columns)

Index(['S', 'D', 'C', 'score'], dtype='object')


In [8]:
config1 = df["S"].tolist()
config2 = df["D"].tolist()
config3 = df["C"].tolist()
config4 = (df["S"].astype(str) + "\n" + df["D"].astype(str)).tolist()
config5 = (df["D"].astype(str) + "\n" + df["C"].astype(str)).tolist()

### Resolving Hugging Face Authentication for Model Download

The `ModuleNotFoundError: No module named 'custom_st'` is likely caused by an incomplete or failed download of a model component from the Hugging Face Hub. While authentication is often optional for public models, providing an `HF_TOKEN` can prevent issues like rate limiting or partial downloads.

**Steps to resolve:**
1.  **Get your Hugging Face Token:**
    *   Go to [Hugging Face Settings/Tokens](https://huggingface.co/settings/tokens).
    *   Create a new token with at least 'read' access.
2.  **Add to Colab Secrets:**
    *   In Google Colab, click the "🔑" (Secrets) icon in the left sidebar.
    *   Add a new secret with the name `HF_TOKEN` and paste your Hugging Face token as the value.
3.  **Configure in Notebook:** Run the following cell to configure the token.
4.  **Restart Runtime:** After setting the token, you *must* restart your Colab runtime (Runtime -> Restart runtime) for the changes to fully take effect.
5.  **Re-run cells:** After restarting, re-run all your cells, including the one that loads the `SentenceTransformer` model.

In [10]:
# Used to securely store your API key
from google.colab import userdata

# Set the Hugging Face token from Colab secrets
import os
HF_TOKEN = userdata.get('HF_TOKEN')

# Ensure the token is set as an environment variable
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("HF_TOKEN successfully loaded from Colab secrets and set as environment variable.")
else:
    print("HF_TOKEN not found in Colab secrets. Please add it.")

HF_TOKEN successfully loaded from Colab secrets and set as environment variable.


In [17]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [39]:
#for ytucosmos and microsoft

import os
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

model = SentenceTransformer(
    "jinaai/jina-embeddings-v5-text-small",
    device="cuda")

# Ensure all elements in config2 are string
config_str = [str(x) for x in config2]

embeddings = model.encode(config_str, batch_size=2, show_progress_bar=True)

# Define the directory path
output_dir = "/content/drive/MyDrive/AI_HW/"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Save the embeddings
np.save(os.path.join(output_dir, "jina_config2.npy"), embeddings)

del model
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/5495 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 9.83 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.68 GiB is free. Including non-PyTorch memory, this process has 12.88 GiB memory in use. Of the allocated memory 12.62 GiB is allocated by PyTorch, and 139.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [42]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B")
model = AutoModel.from_pretrained(
    "Qwen/Qwen3-Embedding-0.6B",
    torch_dtype=torch.float16,
    device_map="cuda"
)
model.eval()

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3Model(
  (embed_tokens): Embedding(151669, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

In [43]:
def get_embeddings(texts, batch_size=4):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, return_tensors="pt").to("cuda")
    return np.vstack(all_embeddings)

In [45]:
config1_str = [str(x) for x in config1]
embeddings = get_embeddings(config1_str, batch_size=4)
np.save("/content/drive/MyDrive/AI_HW/Qwen_config1.npy", embeddings)

In [40]:
del model

import torch
import gc
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 7            |        cudaMalloc retries: 8         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   1971 MiB |  14395 MiB |  96008 MiB |  94037 MiB |
|       from large pool |   1662 MiB |  13778 MiB |  92291 MiB |  90628 MiB |
|       from small pool |    308 MiB |    928 MiB |   3716 MiB |   3408 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   1971 MiB |  14395 MiB |  96008 MiB |  94037 MiB |
|       from large pool |   1662 MiB |  13778 MiB |  92291 MiB |

In [41]:
import torch
print(torch.cuda.memory_allocated() / 1e9, "GB used")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

2.066910208 GB used
2.210398208 GB reserved
